# Sim Flip Chip Distance - Route C - No Inset

This notebook validates the official tutorial asset and stack coverage only. Builder execution is disabled for this fixture until high-count bump contact and lumped-port sheet planning are implemented.

Inset breakpoints: `0, 50 nm, 100 nm, 200 nm, 500 nm, 1 um`.
Sidewall inset bands that would degenerate are skipped by the planner.


In [ ]:
from pathlib import Path
import json

import gdstk
from semantic_geometry_builder import (
    SemanticGeometryBuilder,
    build_gds_stack_geometry_input,
)

GEOMETRY_ID = 'sim_flip_chip_distance'
ROUTE = 'C'
MODE = 'no_inset'
BUILD_SUPPORTED = False
INSET_BREAKPOINTS_UM = [0.0, 0.05, 0.1, 0.2, 0.5, 1.0]
TOP_CELL_NAME = 'sim_flip_chip_distance_fixture'

ROOT = Path.cwd()
while not (ROOT / "pyproject.toml").is_file():
    if ROOT.parent == ROOT:
        raise RuntimeError("Could not locate repository root")
    ROOT = ROOT.parent

ASSET_DIR = ROOT / "tutorials" / "assets"
RUN_FOLDER = ROOT / "tutorials" / "runs" / MODE / GEOMETRY_ID / f"route_{ROUTE.lower()}"
GDS_FILE = ASSET_DIR / f"{GEOMETRY_ID}.gds"
STACK_FILE = ASSET_DIR / f"{GEOMETRY_ID}.stack.json"
METADATA_OVERRIDE = {"inset_routes": []}

build_input = build_gds_stack_geometry_input(
    gds_file=GDS_FILE,
    stack_file=STACK_FILE,
    top_cell_name=TOP_CELL_NAME,
    metadata=METADATA_OVERRIDE,
)
print("polygons:", len(build_input.polygons))
print("entities:", len(build_input.entities))
print("inset routes:", build_input.metadata.get("inset_routes"))


In [ ]:
stack = json.loads(STACK_FILE.read_text())
if TOP_CELL_NAME is not None:
    cell = next(c for c in gdstk.read_gds(str(GDS_FILE)).cells if c.name == TOP_CELL_NAME)
    gds_layers = {(int(p.layer), int(p.datatype)) for p in cell.get_polygons(apply_repetitions=True)}
    layer_records = {(int(r["layer"]), int(r["datatype"])) for r in stack.get("layers", [])}
    ignored = {(int(r["layer"]), int(r["datatype"])) for r in stack["metadata"].get("ignored_layout_layers", [])}
    deferred = {(int(r["layer"]), int(r["datatype"])) for r in stack["metadata"].get("deferred_high_count_layers", [])}
    port_sheets = {(int(r["layer"]), int(r["datatype"])) for r in stack["metadata"].get("port_sheet_source_layers", [])}
    print("unclassified layers:", sorted(gds_layers - layer_records - ignored - deferred - port_sheets))
    print("ignored layers:", sorted(ignored))
    print("port-sheet layers:", sorted(port_sheets))
    print("deferred layers:", sorted(deferred))
    assert (40, 1) in ignored, "Under Bump layer must stay ignored"
    assert (40, 1) not in deferred, "Under Bump layer must not be deferred for simulation"
    assert (202, 1) in port_sheets, "Palace lumped-port sheet layer must be explicit port-sheet source"


In [ ]:
if not BUILD_SUPPORTED:
    print("Builder intentionally not run for this fixture yet.")
else:
    groups = SemanticGeometryBuilder().build(
        build_input,
        route=ROUTE,
        run_folder=RUN_FOLDER,
    )
    xao_path = RUN_FOLDER / "geometry" / f"semantic_geometry_route_{ROUTE.lower()}.xao"
    print("XAO:", xao_path)
    print("groups:", len(groups))
    for group in groups:
        print(group.dimension, group.physical_name, len(group.entity_tags))
    assert xao_path.is_file()
    assert not list(RUN_FOLDER.rglob("*.msh"))
